# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [1]:
   import os
   print(os.getcwd())
   print(os.listdir('.'))
   print(os.listdir('..'))

C:\Users\asus
['.anaconda', '.antigravity-ide', '.bash_history', '.bun', '.cache', '.conda', '.config', '.continuum', '.gemini', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.local', '.webmcp', '01_CNN_Implementation.ipynb', '02_CNN_ModelLoading_Visualization.ipynb', '03_Transfer_Learning_MobileNetV2.ipynb', '333.ipynb', 'anaconda3', 'anaconda_projects', 'Anwendungsdaten', 'AppData', 'code of deeplearning', 'Contacts', 'Cookies', 'data-viz-class-material-w2026', 'dataviz-exercises-maryammoghaddasbayat', 'Desktop', 'Documents', 'Downloads', 'Druckumgebung', 'Eigene Dateien', 'Favorites', 'lecture04_exercise_maryam.ipynb', 'lecture05_exercise_1.ipynb', 'Links', 'Lokale Einstellungen', 'Music', 'Netzwerkumgebung', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{92b49a48-e27a-11f0-8ccd-f068e3bb7525}.TM.blf', 'NTUSER.DAT{92b49a48-e27a-11f0-8ccd-f068e3bb7525}.TM.blf.cnpf', 'NTUSER.DAT{92b49a48-e27a-11f0-8ccd-f068e3bb7525}.TMContainer00000000000000000001.regtra

In [2]:
from pathlib import Path
for p in Path.home().rglob('airbnb_london.csv'):
    print(p)

C:\Users\asus\Documents\data-viz-class-material-w2026\data\airbnb_london.csv
C:\Users\asus\data-viz-class-material-w2026\data\airbnb_london.csv


In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv(r'C:\Users\asus\data-viz-class-material-w2026\data\airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))

Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [6]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

In [ ]:
# Task 1 — overlapping histogram of price by room type
rooms = ['Entire home/apt', 'Private room']
df_t1 = df_cap[df_cap['room_type'].isin(rooms)]

# Colour has meaning: one fixed colour per room type, reused (darker) for its median line
colour_map = {'Entire home/apt': '#1f77b4', 'Private room': '#ff7f0e'}
line_map = {'Entire home/apt': '#08306b', 'Private room': '#a63603'}

medians = df_t1.groupby('room_type')['price'].median()
ratio = medians['Entire home/apt'] / medians['Private room']

fig1 = px.histogram(
    df_t1,
    x='price',
    color='room_type',
    barmode='overlay',
    opacity=0.6,
    nbins=50,
    category_orders={'room_type': rooms},
    color_discrete_map=colour_map,
    labels={'price': 'Price per night (£)', 'room_type': 'Room type'},
    title=(f"Both price distributions are right-skewed, but entire homes sit about "
           f"{ratio:.1f}x higher (median £{medians['Entire home/apt']:.0f} vs £{medians['Private room']:.0f})"),
)

# Median line for EACH room type, differently coloured, with annotation
for i, room in enumerate(rooms):
    fig1.add_vline(
        x=medians[room],
        line_dash='dash',
        line_width=2,
        line_color=line_map[room],
        annotation_text=f"{room} median: £{medians[room]:.0f}",
        annotation_position='top right' if i == 0 else 'top left',
        annotation_font_color=line_map[room],
    )

# Annotate the outlier cap (rule 1)
fig1.add_annotation(
    text=f"Prices capped at the 95th percentile (£{p95:.0f}); more expensive listings excluded",
    xref='paper', yref='paper', x=1, y=-0.2,
    showarrow=False, xanchor='right', font=dict(size=11, color='gray'),
)

fig1.update_layout(
    template='plotly_white',
    legend_title_text='Room type',
    yaxis_title='Number of listings',
    margin=dict(b=90),
)
fig1.show()


## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [ ]:
# Task 2 — horizontal box plot of reviews per month by borough

# Find the borough column (name differs between dataset versions)
borough_col = next(c for c in ['neighbourhood_cleansed', 'neighbourhood', 'borough',
                               'neighborhood', 'neighbourhood_group'] if c in df.columns)

# Zero-review listings are new/inactive: filter them out before plotting
df_t2 = df[df['reviews_per_month'] > 0].dropna(subset=['reviews_per_month', borough_col])

# Order boroughs by median activity
med = df_t2.groupby(borough_col)['reviews_per_month'].median().sort_values(ascending=False)
top2 = list(med.index[:2])
# Plotly draws the first category at the bottom of a horizontal axis,
# so reverse the order to put the most active borough at the top
order = list(med.index[::-1])

# Colour has meaning: highlight the two most active boroughs, grey for the rest
df_t2 = df_t2.assign(group=np.where(df_t2[borough_col].isin(top2), 'Two most active', 'Other boroughs'))

fig2 = px.box(
    df_t2,
    x='reviews_per_month',
    y=borough_col,
    color='group',
    orientation='h',
    points='outliers',  # outliers as individual points
    category_orders={borough_col: order, 'group': ['Two most active', 'Other boroughs']},
    color_discrete_map={'Two most active': '#d62728', 'Other boroughs': '#9aa5b1'},
    labels={'reviews_per_month': 'Reviews per month', borough_col: '', 'group': ''},
    title=f"{top2[0]} and {top2[1]} are London's busiest boroughs, with the highest median reviews per month",
)
fig2.update_traces(boxmean=False)

# Overall median reference line with annotation
overall_med = df_t2['reviews_per_month'].median()
fig2.add_vline(
    x=overall_med, line_dash='dash', line_color='black', line_width=1.5,
    annotation_text=f"All-London median: {overall_med:.2f}",
    annotation_position='top right',
)

fig2.add_annotation(
    text="Listings with zero reviews (new/inactive) excluded. Reviews per month is a proxy for booking frequency.",
    xref='paper', yref='paper', x=1, y=-0.12,
    showarrow=False, xanchor='right', font=dict(size=11, color='gray'),
)

fig2.update_layout(
    template='plotly_white',
    height=max(500, 24 * len(med) + 160),
    legend_title_text='',
    margin=dict(b=90),
)
fig2.show()
